# Chapter 13 — Normalize at the Boundary

**Companion to Applied AI**

Question: What breaks when usage fields with different meanings are silently merged?

By the end of this notebook you will have:

- normalized incompatible usage payloads into one canonical record
- preserved raw payloads, canonical fields, unknowns, and provenance
- shown the danger of zero-filling absent usage fields

## What this notebook demonstrates
Three dialects count different things under similar names. The notebook derives one honest canonical record per call — and refuses to invent missing fields.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import json

seed: 42


## 1. Incompatible usage semantics

In [2]:
payloads = {
    "alpha": {"input_tokens": 38, "output_tokens": 12, "cached": 30},   # input INCLUDES cache
    "beta":  {"prompt_tokens": 279, "completion_tokens": 3},            # no cache info at all
    "gamma": {"fresh_input": 73, "read_input": 14, "output": 9},        # cache reported separately
}
print(json.dumps(payloads, indent=1))

{
 "alpha": {
  "input_tokens": 38,
  "output_tokens": 12,
  "cached": 30
 },
 "beta": {
  "prompt_tokens": 279,
  "completion_tokens": 3
 },
 "gamma": {
  "fresh_input": 73,
  "read_input": 14,
  "output": 9
 }
}


## 2. Normalize: equivalent merged, related preserved, rest UNKNOWN

In [3]:
def normalize(name: str, p: dict) -> dict:
    raw = dict(p)
    if name == "alpha":
        rec = {"fresh_input": p["input_tokens"] - p["cached"], "output": p["output_tokens"],
               "cache_detail": f"cached={p['cached']}", "provenance": name}
    elif name == "beta":
        rec = {"fresh_input": "UNKNOWN", "output": p["completion_tokens"],
               "cache_detail": "UNKNOWN", "provenance": name}
    else:
        rec = {"fresh_input": p["fresh_input"], "output": p["output"],
               "cache_detail": f"read={p['read_input']}", "provenance": name}
    rec["raw_payload"] = raw
    return rec

for name, p in payloads.items():
    print(name, "->", normalize(name, p))
assert normalize("beta", payloads["beta"])["fresh_input"] == "UNKNOWN"
assert all(r["raw_payload"] is not None for r in (normalize(k, v) for k, v in payloads.items()))

alpha -> {'fresh_input': 8, 'output': 12, 'cache_detail': 'cached=30', 'provenance': 'alpha', 'raw_payload': {'input_tokens': 38, 'output_tokens': 12, 'cached': 30}}
beta -> {'fresh_input': 'UNKNOWN', 'output': 3, 'cache_detail': 'UNKNOWN', 'provenance': 'beta', 'raw_payload': {'prompt_tokens': 279, 'completion_tokens': 3}}
gamma -> {'fresh_input': 73, 'output': 9, 'cache_detail': 'read=14', 'provenance': 'gamma', 'raw_payload': {'fresh_input': 73, 'read_input': 14, 'output': 9}}


## 3. Break it: zero-fill the unknown and watch cost math lie

In [4]:
honest = normalize("beta", payloads["beta"])
lied = dict(honest, fresh_input=0)  # the tempting silent default
price = 2.50 / 1e6
print("honest fresh-input cost: UNKNOWN (refuses to compute)")
print(f"zero-filled fresh-input cost: ${lied['fresh_input'] * price:.6f} <- looks free, is not measured")
assert honest["fresh_input"] == "UNKNOWN"

honest fresh-input cost: UNKNOWN (refuses to compute)
zero-filled fresh-input cost: $0.000000 <- looks free, is not measured


## Interpretation
- Supports: normalize equivalent fields, preserve related ones, keep the rest explicitly unknown — with the raw payload retained.
- Does NOT support: cross-vendor cost comparisons from these toy numbers.

## Try it yourself
1. Add a dialect reporting only a `total` and normalize it to all-UNKNOWN with raw preserved.
2. Write a `fresh_input_cost` function that raises on UNKNOWN instead of computing.
3. Hash the canonical records and show the hash is stable across reruns.